# MCP agent scratch notebook

Small Monte Carlo π estimate for exercising `run` / `manifest.jsonl`.


In [1]:
import numpy as np
rng = np.random.default_rng(42)


In [2]:
n = 8000
xy = rng.random((n, 2))
inside = (xy[:, 0]**2 + xy[:, 1]**2) <= 1.0
pi_est = 4 * inside.mean()


In [3]:
print("pi_estimate", float(pi_est), "samples", int(n))


pi_estimate 3.166 samples 8000


## Emitter localisation toy (ranges)

Three anchors in the plane; noisy squared Euclidean ranges; iterative Gauss-Newton updates with printed residuals.


In [4]:
anchors = np.array([[0.,0.],[1.,0.],[0.,1.]], float)
true_xy = np.array([0.27, 0.41])
sigma = 0.05
meas = np.linalg.norm(anchors - true_xy, axis=1) ** 2
meas = meas + rng.normal(0., sigma, size=meas.shape)
print("measured_squared_ranges", meas.round(4).tolist())


measured_squared_ranges [0.2193, 0.8067, 0.3869]


In [5]:
guess = np.array([0.5, 0.5])

for step in range(8):
    r = anchors - guess
    pred_sq = np.sum(r**2, axis=1)
    res = meas - pred_sq
    print(f"step {step}: guess={guess}, max|residual|={np.max(np.abs(res)):.6f}")
    J = -2.0 * r
    delta, *_ = np.linalg.lstsq(J, res, rcond=None)
    guess = guess + 0.35 * delta

print("final_guess", guess, "truth", true_xy, "RMSE_truth", float(np.linalg.norm(guess - true_xy)))


step 0: guess=[0.5 0.5], max|residual|=0.306702
step 1: guess=[0.41415052 0.4876142 ], max|residual|=0.225715
step 2: guess=[0.34824683 0.47105235], max|residual|=0.160029
step 3: guess=[0.30065194 0.4554652 ], max|residual|=0.110166
step 4: guess=[0.26802182 0.4431736 ], max|residual|=0.074507
step 5: guess=[0.24634029 0.43435585], max|residual|=0.050034
step 6: guess=[0.23213214 0.42831769], max|residual|=0.033625
step 7: guess=[0.22286377 0.42427117], max|residual|=0.022755
final_guess [0.21682232 0.42158607] truth [0.27 0.41] RMSE_truth 0.05442520044827225
